# 02 · ETL e Integração SIH + CNES

**Objetivo:** Carregar os dados traduzidos do SIH e CNES, filtrar internações por IAM (CID I21), realizar limpeza e integrar as duas bases em uma base de modelagem unificada.

**Inputs:**
- `data/interim/sih_*_traduzido.csv` — registros de AIH do SIH com colunas `_DESC` de códigos traduzidos
- `data/interim/cnes_*_traduzido.csv` — tabelas do CNES (ST, LT, EQ, SR, HB) com colunas `_DESC` traduzidas
- `data/external/dicionario_SIH.json` — mapeamento de nomes de colunas do SIH
- `data/external/dicionario_CNES_*.json` — mapeamentos de nomes de colunas do CNES

**Outputs gerados:**
- `data/interim/sih_iam.csv` — internações por IAM (base limpa SIH)
- `data/interim/cnes_hospitais.csv` — base mestre de hospitais (CNES consolidado)
- `data/processed/base_modelagem.csv` — base final SIH × CNES pronta para modelagem

## 0. Configuração do Ambiente

In [18]:
import pandas as pd
import numpy as np
import os
import json
from pathlib import Path
import sys

In [19]:
# Adiciona a raiz do projeto ao sys.path para importar src/
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print(f"Raiz do projeto: {ROOT}")

Raiz do projeto: c:\dsm\tcc


In [20]:
pd.set_option('display.float_format', '{:.2f}'.format)

# ── Caminhos ───────────────────────────────────────────────────────────
# Os CSVs de entrada agora são os arquivos *traduzidos* gerados pelo
# notebook 01_data_collection.ipynb e salvos em data/interim/
INTERIM   = Path(ROOT, 'data', 'interim')    # traduzidos (input) e outputs do ETL
PROCESSED = Path(ROOT, 'data', 'processed')
EXTERNAL  = Path(ROOT, 'data', 'external')

INTERIM.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

print("Caminhos configurados:")
print(f"  Entrada: {INTERIM}")
print(f"  Saída  : {PROCESSED}")

Caminhos configurados:
  Entrada: c:\dsm\tcc\data\interim
  Saída  : c:\dsm\tcc\data\processed


## 1. Carregamento e Padronização do SIH

### 1.1 Leitura dos arquivos

Concatenamos **todos** os arquivos CSV disponíveis em `data/input/SIH/`.
O dicionário `dicionario_SIH.json` é usado para renomear as colunas para nomes legíveis.

In [21]:
# Lê os CSVs traduzidos do SIH (padrão: sih_*_traduzido.csv)
arquivos_sih = sorted(INTERIM.glob('sih_*_traduzido.csv'))
print(f"Arquivos SIH traduzidos encontrados: {len(arquivos_sih)}")
for f in arquivos_sih:
    print(f"  {f.name}")

Arquivos SIH traduzidos encontrados: 1
  sih_rdsp2501_traduzido.csv


In [22]:
# Carrega e renomeia colunas usando o dicionário externo
path_dicionario = Path(ROOT, 'data', 'external', 'dicionario_SIH.json')
with open(path_dicionario, 'r', encoding='utf-8') as f:
    schema = json.load(f)

rename_dict = {col["old_name"]: col["new_name"] for col in schema}

# Concatena todos os meses disponíveis
frames = []
for arq in arquivos_sih:
    df_tmp = pd.read_csv(arq, dtype=str, low_memory=False)
    df_tmp = df_tmp.rename(columns=rename_dict)
    frames.append(df_tmp)

df_sih = pd.concat(frames, ignore_index=True)
print(f"SIH carregado: {df_sih.shape[0]:,} registros × {df_sih.shape[1]} colunas")
df_sih.head(3)

SIH carregado: 238,144 registros × 169 colunas


,municipio_gestor,ano_competencia,mes_competencia,especialidade_leito,cnpj_hospital,numero_aih,tipo_aih,cep_paciente,municipio_residencia,data_nascimento,...,MARCA_UCI_DESC,TPDISEC1_DESC,TPDISEC2_DESC,TPDISEC3_DESC,TPDISEC4_DESC,TPDISEC5_DESC,TPDISEC6_DESC,TPDISEC7_DESC,TPDISEC8_DESC,TPDISEC9_DESC
0,350000,2025,1,1,46374500028366.0,3525100117847,1,11704840,354100,19840716,...,NaN,Preexistente,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,350000,2025,1,2,46374500028366.0,3524130275908,1,11741802,352210,20070606,...,NaN,Preexistente,Preexistente,Preexistente,Preexistente,Preexistente,NaN,NaN,NaN,NaN
2,350000,2025,1,2,46374500028366.0,3524130278427,1,11730000,353110,20030120,...,NaN,Preexistente,Preexistente,Preexistente,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
# identificação dos tipos das colunas
df_sih.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 238144 entries, 0 to 238143
Columns: 169 entries, municipio_gestor to TPDISEC9_DESC
dtypes: object(169)
memory usage: 307.1+ MB


Todas  as colunas foram identificadas como tipo string, o que devera ser ajustado em breve.

### 1.2 Filtro CID — Internações por IAM

Filtramos registros cujo diagnóstico principal **ou** secundário inicie com `I21`
(Infarto Agudo do Miocárdio — CID-10), que é o foco do estudo.

In [24]:
df_sih["diagnostico_principal"]  = df_sih["diagnostico_principal"].astype(str)
df_sih["diagnostico_secundario"] = df_sih["diagnostico_secundario"].astype(str)

df_iam = df_sih[
    df_sih["diagnostico_principal"].str.startswith("I21") |
    df_sih["diagnostico_secundario"].str.startswith("I21")
].copy()

print(f"Total de internações: {len(df_sih)}")
print(f"Internações por IAM : {len(df_iam)} ({len(df_iam)/len(df_sih)*100:.1f}%)")

Total de internações: 238144
Internações por IAM : 4110 (1.7%)


### 1.2 Seleção de Colunas Relevantes

In [25]:
df_iam.shape

(4110, 169)

#### Tratamento de Valores Ausentes
Campos administrativos do DataSUS frequentemente chegam como strings vazias ou `"0000"`.
Substituímos apenas **colunas de texto** (object), preservando zeros em variáveis
numéricas legítimas (ex.: `indicador_obito=0` = alta; `uti_mes_total=0` = sem UTI).

In [26]:
# Aplica replace somente em colunas de texto para não corromper variáveis numéricas/binárias
str_cols = df_iam.select_dtypes(include='object').columns
df_iam[str_cols] = df_iam[str_cols].replace(["", "0000", "000"], pd.NA)

# Converte colunas numéricas para tipo correto
num_cols = ["idade", "dias_permanencia", "indicador_obito",
            "uti_mes_total", "codigo_idade"]
for col in num_cols:
    if col in df_iam.columns:
        df_iam[col] = pd.to_numeric(df_iam[col], errors='coerce')

        # IDADE INCOERentE

print("Tratamento de nulos concluído.")

Tratamento de nulos concluído.


C:\Users\carol\AppData\Local\Temp\ipykernel_21392\2883878836.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_iam[str_cols] = df_iam[str_cols].replace(["", "0000", "000"], pd.NA)


Ainda temos 114 colunas e nem todas serão nesessárias então iremos aplicar uma limpeza.
#### Remoção de colunas inutilizáveis

In [27]:
pct_nulos = df_iam.isnull().mean() * 100
print(pct_nulos.sort_values(ascending=False))

TPDISEC9_DESC        100.00
DT_SAIDA_DESC        100.00
INSTRU_DESC          100.00
NUM_FILHOS_DESC      100.00
MORTE_DESC           100.00
                      ...  
carater_internacao     0.00
total_pontos_sp        0.00
indicador_homonimo     0.00
tipo_uci               0.00
municipio_gestor       0.00
Length: 169, dtype: float64


In [28]:
# ── Remoção de colunas inutilizáveis ─────────────────────────────────────────
# Colunas 100% nulas (sem informação alguma)
limite = 70  # %
cols_remover = pct_nulos[pct_nulos > limite].index
df_iam = df_iam.drop(columns=cols_remover)
print(f'Removidas {len(cols_remover)} colunas com mais de {limite}% de nulos')


# Filtro de idade: manter apenas adultos (>= 18 anos)
#    IAM pediátrico é evento raro e biologicamente distinto — excluído do escopo
df_iam['idade'] = pd.to_numeric(df_iam['idade'], errors='coerce')
n_antes = len(df_iam)
df_iam = df_iam[df_iam['idade'] >= 18].copy()
print(f'Registros pediátricos removidos (idade < 18): {n_antes - len(df_iam)}')
print(f'Base SIH-IAM adultos: {len(df_iam)} registros × {df_iam.shape[1]} colunas')

Removidas 59 colunas com mais de 70% de nulos
Registros pediátricos removidos (idade < 18): 1
Base SIH-IAM adultos: 4109 registros × 110 colunas


In [29]:
# Resumo de completude
nulls = pd.DataFrame({
    "qtd_nulos":  df_iam.isnull().sum(),
    "perc_nulos": df_iam.isnull().mean() * 100
}).sort_values("perc_nulos", ascending=False)

nulls[nulls["qtd_nulos"] > 0]

,qtd_nulos,perc_nulos
cnpj_mantenedora,2194,53.39
TPDISEC1_DESC,2136,51.98
diagnostico_secundario_1,2136,51.98
COBRANCA_DESC,1090,26.53
DIAS_PERM_DESC,821,19.98
cnpj_hospital,618,15.04
etnia,65,1.58


### 1.3 Salvar Base SIH-IAM Intermediária

In [30]:
path_sih_interim = INTERIM / "sih_iam.csv"
df_iam.to_csv(path_sih_interim, index=False)
print(f"SIH-IAM salvo em: {path_sih_interim}  ({len(df_iam):,} registros)")

SIH-IAM salvo em: c:\dsm\tcc\data\interim\sih_iam.csv  (4,109 registros)


## 2. Carregamento e Padronização do CNES

Carregamos as cinco tabelas do CNES usando dicionários JSON específicos para cada prefixo.

In [31]:
# Lê os CSVs traduzidos do CNES (padrão: cnes_*_traduzido.csv)
arquivos_cnes = sorted(INTERIM.glob('cnes_*_traduzido.csv'))
print(f"Arquivos CNES traduzidos encontrados: {len(arquivos_cnes)}")
for f in arquivos_cnes:
    print(f"  {f.name}")

Arquivos CNES traduzidos encontrados: 5
  cnes_eq_eqsp2512_traduzido.csv
  cnes_hb_hbsp2512_traduzido.csv
  cnes_lt_ltsp2512_traduzido.csv
  cnes_sr_srsp2512_traduzido.csv
  cnes_st_stsp2512_traduzido.csv


In [32]:
def load_cnes_custom(tipo, arquivos_cnes):
    tipo = tipo.lower()

    # Localiza o arquivo traduzido com prefixo cnes_{tipo}_
    arq = next(
        (f for f in arquivos_cnes if f.name.lower().startswith(f'cnes_{tipo}_')),
        None
    )

    if arq is None:
        raise FileNotFoundError(
            f"Arquivo traduzido cnes_{tipo}_*_traduzido.csv não encontrado."
            f" Arquivos disponíveis: {[f.name for f in arquivos_cnes]}"
        )

    # Dicionário de renomeação de colunas (old_name → new_name)
    path_dict = Path(ROOT, 'data', 'external', f'dicionario_CNES_{tipo.upper()}.json')
    with open(path_dict, 'r', encoding='utf-8') as f:
        schema = json.load(f)

    rename_dict = {col["old_name"]: col["new_name"] for col in schema}

    df = pd.read_csv(arq, dtype=str, low_memory=False)

    # Renomeia as colunas originais; as colunas _DESC ficam com o nome gerado
    # automaticamente (ex: TP_LEITO_DESC → mantida como está)
    df = df.rename(columns=rename_dict)

    df['arquivo_origem'] = arq.name
    df['tipo_cnes'] = tipo.upper()

    return df

In [33]:
df_st = load_cnes_custom('st', arquivos_cnes)
df_lt = load_cnes_custom('lt', arquivos_cnes)
df_eq = load_cnes_custom('eq', arquivos_cnes)
df_sr = load_cnes_custom('sr', arquivos_cnes)
df_hb = load_cnes_custom('hb', arquivos_cnes)

print(f"\nRegistros carregados por tabela:")
print(f"  ST (Estabelecimentos) : {df_st.shape}")
print(f"  LT (Leitos)           : {df_lt.shape}")
print(f"  EQ (Equipamentos)     : {df_eq.shape}")
print(f"  SR (Serviços)         : {df_sr.shape}")
print(f"  HB (Habilitações)     : {df_hb.shape}")


Registros carregados por tabela:
  ST (Estabelecimentos) : (109849, 280)
  LT (Leitos)           : (8350, 43)
  EQ (Equipamentos)     : (242363, 44)
  SR (Serviços)         : (177050, 47)
  HB (Habilitações)     : (6755, 52)


### 2.1 pivot Tabelas CNES 

#### Remoção de Duplicidades (ST)

In [34]:
# Garante uma linha por hospital — mantém o registro mais recente
n_antes = len(df_st)
df_st = df_st.drop_duplicates(subset=['codigo_cnes'], keep='last')
print(f"ST: {n_antes} → {len(df_st)} registros únicos (removidas {n_antes - len(df_st):,} duplicatas)")

ST: 109849 → 109849 registros únicos (removidas 0 duplicatas)


#### Agregação de Leitos (LT)

In [35]:
# Pivot de leitos por tipo
df_lt['quantidade_leitos_existentes'] = pd.to_numeric(
    df_lt['quantidade_leitos_existentes'],
    errors='coerce'
).fillna(0)

df_lt_pivot = df_lt.pivot_table(
    index='codigo_cnes',
    columns='tipo_leito',
    values='quantidade_leitos_existentes',
    aggfunc='sum',
    fill_value=0
)

df_lt_pivot.columns = [
    f'leitos_{col}'
    for col in df_lt_pivot.columns
]

df_lt_pivot = df_lt_pivot.reset_index()

#### Agregação de Equipamentos (EQ)

In [36]:
# Pivot de equipamentos
df_eq['quantidade_em_uso'] = pd.to_numeric(
    df_eq['quantidade_em_uso'],
    errors='coerce'
).fillna(0)

df_eq_pivot = df_eq.pivot_table(
    index='codigo_cnes',
    columns='codigo_equipamento',
    values='quantidade_em_uso',
    aggfunc='sum',
    fill_value=0
)

df_eq_pivot.columns = [
    f'equip_{col}'
    for col in df_eq_pivot.columns
]

df_eq_pivot = df_eq_pivot.reset_index()

#### Serviços Especializados (SR) e Habilitações (HB)

In [37]:
# Serviços especializados
df_sr['flag_servico'] = 1

df_sr_pivot = df_sr.pivot_table(
    index='codigo_cnes',
    columns='codigo_servico_especializado',
    values='flag_servico',
    aggfunc='max',
    fill_value=0
)

df_sr_pivot.columns = [
    f'servico_{col}'
    for col in df_sr_pivot.columns
]

df_sr_pivot = df_sr_pivot.reset_index()

In [38]:
# Habilitações
df_hb['flag_habilitacao'] = 1

df_hb_pivot = df_hb.pivot_table(
    index='codigo_cnes',
    columns='codigo_habilitacao',
    values='flag_habilitacao',
    aggfunc='max',
    fill_value=0
)

df_hb_pivot.columns = [
    f'habilitacao_{col}'
    for col in df_hb_pivot.columns
]

df_hb_pivot = df_hb_pivot.reset_index()

In [39]:
print(df_lt_pivot.shape)
print(df_eq_pivot.shape)
print(df_sr_pivot.shape)
print(df_hb_pivot.shape)

(1462, 8)
(50460, 99)
(41011, 64)
(2282, 190)


## 3 Salvar Base Mestre de Hospitais (Interim CNES)

In [40]:
# ── Merge das tabelas CNES em torno da ST (uma linha por hospital) ───────────
#
# ST  : tabela mestre — um registro por hospital (já deduplicada acima)
# LT  : leitos por tipo de leito  → pivotado em df_lt_pivot
# EQ  : equipamentos por código   → pivotado em df_eq_pivot
# SR  : serviços especializados   → pivotado em df_sr_pivot
# HB  : habilitações              → pivotado em df_hb_pivot
#
# Usamos left join para manter todos os hospitais da ST,
# mesmo que não possuam leitos, equipamentos, etc. registrados.

df_cnes_final = df_st.copy()

for df_pivot, nome in [
    (df_lt_pivot, 'LT — Leitos'),
    (df_eq_pivot, 'EQ — Equipamentos'),
    (df_sr_pivot, 'SR — Serviços Especializados'),
    (df_hb_pivot, 'HB — Habilitações'),
]:
    # Garante que a chave está no mesmo formato (string sem espaços)
    df_pivot['codigo_cnes'] = df_pivot['codigo_cnes'].astype(str).str.strip()
    df_cnes_final = pd.merge(df_cnes_final, df_pivot, on='codigo_cnes', how='left')
    print(f'  Após merge {nome}: {df_cnes_final.shape}')

# Preenche NaN nas colunas de contagem/flag com 0
# (hospitais sem leitos/equipamentos/serviços registrados ficam como 0, não NaN)
cols_pivot = (
    list(df_lt_pivot.columns.drop('codigo_cnes')) +
    list(df_eq_pivot.columns.drop('codigo_cnes')) +
    list(df_sr_pivot.columns.drop('codigo_cnes')) +
    list(df_hb_pivot.columns.drop('codigo_cnes'))
)
df_cnes_final[cols_pivot] = df_cnes_final[cols_pivot].fillna(0)

# Remove colunas auxiliares que vieram das tabelas secundárias e são redundantes
# (arquivo_origem e tipo_cnes existem em cada pivot — causam sufixos _x/_y)
cols_remover_aux = [c for c in df_cnes_final.columns if c.endswith(('_x', '_y'))]
if cols_remover_aux:
    df_cnes_final = df_cnes_final.drop(columns=cols_remover_aux)
    print(f'  Colunas auxiliares duplicadas removidas: {cols_remover_aux}')

print(f'\nBase CNES final: {df_cnes_final.shape[0]:,} hospitais × {df_cnes_final.shape[1]} colunas')


  Após merge LT — Leitos: (109849, 287)
  Após merge EQ — Equipamentos: (109849, 385)
  Após merge SR — Serviços Especializados: (109849, 448)
  Após merge HB — Habilitações: (109849, 637)

Base CNES final: 109,849 hospitais × 637 colunas


### 3.1 Limpeza de Colunas com Muitos Nulos

Assim como feito com o SIH, removemos colunas com alto percentual de valores ausentes.
Colunas com `> 70%` de nulos não carregam informação útil para a modelagem e apenas
adicionam ruído e custo computacional.

**Tipos de colunas que tendem a ser removidas:**
- Colunas `_DESC` sem dicionário `.cnv` mapeado (ficam 100% nulas)
- Campos de controle administrativo do CNES nunca preenchidos em SP
- Campos de contrato municipal/estadual que só existem em alguns estados


In [41]:
# ── Inspeção de nulos no CNES após merge ─────────────────────────────────────
pct_nulos_cnes = df_cnes_final.isnull().mean() * 100

# Distribuição por faixa
faixas = {
    '100%'  : (pct_nulos_cnes == 100).sum(),
    '70–99%': ((pct_nulos_cnes >= 70) & (pct_nulos_cnes < 100)).sum(),
    '50–69%': ((pct_nulos_cnes >= 50) & (pct_nulos_cnes < 70)).sum(),
    '1–49%' : ((pct_nulos_cnes > 0)  & (pct_nulos_cnes < 50)).sum(),
    '0%'    : (pct_nulos_cnes == 0).sum(),
}
print(f'Shape antes da limpeza: {df_cnes_final.shape}')
print('\nDistribuição de nulos por coluna:')
for faixa, qtd in faixas.items():
    print(f'  {faixa:8}: {qtd:>4} colunas')

print('\nTop 20 colunas com mais nulos:')
print(pct_nulos_cnes.sort_values(ascending=False).head(20).to_string())


Shape antes da limpeza: (109849, 637)

Distribuição de nulos por coluna:
  100%    :   48 colunas
  70–99%  :   34 colunas
  50–69%  :    3 colunas
  1–49%   :    8 colunas
  0%      :  544 colunas

Top 20 colunas com mais nulos:
DT_EXPED_DESC              100.00
numero_contrato_estadual   100.00
avaliado_acreditacao       100.00
classificacao_avaliacao    100.00
data_acreditacao           100.00
avaliado_pnass             100.00
data_avaliacao_pnass       100.00
CODUFMUN_DESC              100.00
AP03CV03_DESC              100.00
COD_IR_DESC                100.00
TPGESTAO_DESC              100.00
ESFERA_A_DESC              100.00
RETENCAO_DESC              100.00
AP03CV01_DESC              100.00
AP02CV01_DESC              100.00
CLIENTEL_DESC              100.00
TURNO_AT_DESC              100.00
NIV_HIER_DESC              100.00
TP_PREST_DESC              100.00
DT_PUBLM_DESC              100.00


In [42]:
# ── Remoção de colunas com > 70% de nulos ────────────────────────────────────
LIMITE_NULOS = 70  # mesmo critério aplicado ao SIH

cols_remover_cnes = pct_nulos_cnes[pct_nulos_cnes > LIMITE_NULOS].index.tolist()

df_cnes_final = df_cnes_final.drop(columns=cols_remover_cnes)

print(f'Colunas removidas (> {LIMITE_NULOS}% nulos): {len(cols_remover_cnes)}')
print(f'Shape após limpeza: {df_cnes_final.shape}')
print()

# Resumo de completude após limpeza
nulls_cnes = pd.DataFrame({
    'qtd_nulos' : df_cnes_final.isnull().sum(),
    'perc_nulos': df_cnes_final.isnull().mean() * 100
}).sort_values('perc_nulos', ascending=False)

restantes_com_nulos = nulls_cnes[nulls_cnes['qtd_nulos'] > 0]
print(f'Colunas restantes com algum nulo: {len(restantes_com_nulos)}')
if not restantes_com_nulos.empty:
    display(restantes_com_nulos)


Colunas removidas (> 70% nulos): 82
Shape após limpeza: (109849, 555)

Colunas restantes com algum nulo: 11


,qtd_nulos,perc_nulos
SERAP01P_DESC,65388,59.53
RES_BIOL_DESC,60723,55.28
codigo_regiao_saude,60373,54.96
data_expedicao_alvara,23217,21.14
orgao_expedidor_alvara,22222,20.23
numero_alvara,21886,19.92
RES_COMU_DESC,12808,11.66
NIV_DEP_DESC,12272,11.17
TP_UNID_DESC,4731,4.31
codigo_fluxo_clientela,941,0.86


In [43]:
path_cnes_interim = INTERIM / "cnes_hospitais.csv"
df_cnes_final.to_csv(path_cnes_interim, index=False)
print(f"CNES Interim salvo em: {path_cnes_interim}  ({len(df_cnes_final):,} hospitais)")

CNES Interim salvo em: c:\dsm\tcc\data\interim\cnes_hospitais.csv  (109,849 hospitais)


## 4. Fusão Global: SIH-IAM × CNES

Fazemos um *Left Join* das internações por IAM com a base mestre de hospitais.
A chave de junção é `codigo_cnes`, normalizada em ambos os lados para evitar
divergências por zeros à esquerda ou espaços.

In [44]:
# ── Padroniza a chave de merge nos dois DataFrames ──────────────────────────
# No SIH-IAM o campo foi salvo como 'cnes' (nome original do DATASUS);
# no CNES-final a coluna já foi renomeada para 'codigo_cnes' pelo dicionário.
if 'cnes' in df_iam.columns and 'codigo_cnes' not in df_iam.columns:
    df_iam = df_iam.rename(columns={'cnes': 'codigo_cnes'})

df_iam['codigo_cnes']        = df_iam['codigo_cnes'].astype(str).str.strip().str.zfill(7)
df_cnes_final['codigo_cnes'] = df_cnes_final['codigo_cnes'].astype(str).str.strip().str.zfill(7)

# ── Resolve conflitos de nome entre SIH e CNES antes do merge ────────────────
# Colunas que existem nos dois lados (exceto a chave) receberiam sufixo _x/_y.
# Estratégia: renomear as colunas do CNES adicionando prefixo 'cnes_' quando conflitam.
chave = 'codigo_cnes'
cols_sih  = set(df_iam.columns) - {chave}
cols_cnes = set(df_cnes_final.columns) - {chave}
conflitos = cols_sih & cols_cnes

if conflitos:
    print(f'Colunas em conflito renomeadas no CNES (prefixo cnes_): {sorted(conflitos)}')
    df_cnes_final = df_cnes_final.rename(
        columns={c: f'cnes_{c}' for c in conflitos}
    )

# ── Left Join: mantém todos os registros IAM ──────────────────────────────────
df_base_modelagem = pd.merge(df_iam, df_cnes_final, on=chave, how='left')

# ── Log de qualidade do merge ─────────────────────────────────────────────────
# Usa uma coluna que SÓ existe no CNES para medir o match rate
col_check = 'tipo_unidade'  # sempre vem do CNES-ST, nunca do SIH
n_sem_cnes = df_base_modelagem[col_check].isna().sum()

print(f'Total de internações IAM        : {len(df_base_modelagem):,}')
print(f'Sem correspondência no CNES     : {n_sem_cnes:,}  ({n_sem_cnes/len(df_base_modelagem)*100:.1f}%)')
print(f'Com dados hospitalares do CNES  : {len(df_base_modelagem) - n_sem_cnes:,}')
print(f'Número de features da base final: {df_base_modelagem.shape[1]}')


Colunas em conflito renomeadas no CNES (prefixo cnes_): ['cnpj_mantenedora', 'natureza_juridica', 'tipo_gestao']
Total de internações IAM        : 4,109
Sem correspondência no CNES     : 0  (0.0%)
Com dados hospitalares do CNES  : 4,109
Número de features da base final: 664


In [45]:
# Exporta base de modelagem final
path_base_final = PROCESSED / "base_modelagem.csv"
df_base_modelagem.to_csv(path_base_final, index=False)
print(f"Base de modelagem salva em: {path_base_final}")
df_base_modelagem.sample(3)

Base de modelagem salva em: c:\dsm\tcc\data\processed\base_modelagem.csv


,municipio_gestor,ano_competencia,mes_competencia,especialidade_leito,cnpj_hospital,numero_aih,tipo_aih,cep_paciente,municipio_residencia,data_nascimento,...,habilitacao_807,habilitacao_815,habilitacao_901,habilitacao_902,habilitacao_903,habilitacao_904,habilitacao_905,habilitacao_906,habilitacao_907,habilitacao_908
713,350000,2025,1,1,46374500014730.0,3525101947092,1,8775340,353060,19690211,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2564,352340,2025,1,3,50119585000131.0,3525111532668,1,13254201,352340,19531230,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2361,351830,2025,1,3,48517932000132.0,3524126518374,1,8900000,351830,19570728,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


In [46]:
df_base_modelagem

,municipio_gestor,ano_competencia,mes_competencia,especialidade_leito,cnpj_hospital,numero_aih,tipo_aih,cep_paciente,municipio_residencia,data_nascimento,...,habilitacao_807,habilitacao_815,habilitacao_901,habilitacao_902,habilitacao_903,habilitacao_904,habilitacao_905,habilitacao_906,habilitacao_907,habilitacao_908
0,350000,2025,1,1,60003761000129.0,3524128896860,1,16309146,353730,19740525,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
1,350000,2025,1,1,60003761000129.0,3524128896871,1,15450959,353400,19631015,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2,350000,2025,1,1,60003761000129.0,3524131709505,1,15230970,350020,19550715,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
3,350000,2025,1,1,60003761000129.0,3524128891019,1,15260000,353960,19510501,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
4,350000,2025,1,1,60003761000129.0,3524128891437,1,15200000,352570,19600521,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4104,355670,2025,1,3,72909179000105.0,3524132021510,1,13289058,355670,19360415,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
4105,355670,2025,1,3,72909179000105.0,3524132019287,1,13285754,355670,19560320,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
4106,355670,2025,1,3,72909179000105.0,3524132021982,1,13283196,355670,19731105,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
4107,355700,2025,1,3,NaN,3525109543461,1,18116170,355700,19590614,...,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00


---
## Resumo do Pipeline

| Etapa | Input | Output | Registros |
|-------|-------|--------|-----------|
| 1. Filtro IAM (SIH) | `data/input/SIH/*.csv` | `data/interim/sih_iam.csv` | ver acima |
| 2. Consolidação CNES | `data/input/CNES/*.csv` | `data/interim/cnes_hospitais.csv` | ver acima |
| 3. Fusão global | sih_iam + cnes_hospitais | `data/processed/base_modelagem.csv` | ver acima |

> **Próximos passos:** notebook `03_analise_exploratotia_visualizacao.ipynb` — análise exploratória e visualizações.